# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rowan-ali/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*



The FlyRank research paper emphasizes direct portfolio comparisons as its primary evidence base and treats the machine-learning analyses as exploratory appendix material. The paper also explicitly distinguishes observational findings from causal conclusions.

## Finding 1 — The Freshness Multiplier

The paper reports that the 31–90 day freshness window is the strongest stable freshness band. It also reports that, among 365+ day content, pages refreshed within 30 days had a measured 3.2× higher health score and 57× more impressions than the comparison shown in the study. The paper appropriately notes that the evidence is observational and cautions against treating the result as proof that age or refreshing alone causes the observed improvement.

### My methodology question

A constructive methodology question I would ask is: **How is the outcome being defined and measured, and are the refreshed and unrefreshed pages comparable before the refresh?**

In particular, the reported health score is a FlyRank composite metric built from impressions, position, CTR, and scroll depth rather than a single external outcome measure. It would therefore be useful to understand whether the comparison between refreshed and unrefreshed pages controls for important pre-existing differences, such as prior visibility, content quality, topic demand, or page history.

This question does not challenge the reported finding. It helps clarify how much of the measured difference can be associated with the refresh itself versus differences that already existed between the groups. The safest interpretation is therefore that the paper observed a strong association between recent refreshing and stronger measured performance in this portfolio, rather than establishing a causal effect.

---

## Finding 2 — AI Model Performance

The paper compares OpenAI and Gemini content cohorts using age-controlled comparisons. It reports that each provider family leads in some age windows and concludes that the evidence does not support a blanket claim that one model family universally performs better. The paper describes this analysis as exploratory and recommends comparing content systems within the same age and topic cohorts.

### My methodology question

A constructive methodology question I would ask is: **Does the validation design control for all of the major factors needed to support the scope of the model comparison?**

The paper controls for content age in this comparison, which is an important improvement over comparing cohorts with different age distributions. However, it would also be useful to understand whether other factors, such as topic, intent, publishing period, editing process, and content characteristics, are sufficiently balanced between the provider cohorts.

This matters because a difference in measured health, impressions, or position could reflect differences in the composition of the cohorts rather than a difference caused by the underlying AI model. A matched or more systematically controlled evaluation could provide stronger evidence for a direct provider comparison.

The paper's narrower conclusion is therefore appropriate: the observed results vary across cohorts, and the available evidence does not support a universal claim that one model family wins everywhere.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*


### Validation design

The Week-5 model predicts whether eligible content will **go dark in March 2026**, where `went_dark = 1` means the client-content pair recorded zero Google Search Console clicks during March.

The model uses five signals from the **February 2026 feature window**:

* `gsc_impressions`
* `gsc_avg_position`
* `ga4_sessions`
* `ga4_users`
* `ga4_engaged_sessions`

The March outcome is kept separate from the February feature inputs. The final Week-5 modeling population contains one observation per eligible client-content pair.

For this audit, I compare two validation designs using the same final modeling population and the same Logistic Regression pipeline:

1. **Before — row-level reference split:** rows are randomly divided into training and test sets. Because the same client can appear in both partitions, this design can allow client-specific patterns to be shared across the evaluation boundary.
2. **After — client-grouped split:** all observations belonging to a client are kept entirely within either training or test. This is the preferred design because it evaluates the model on clients that were not used during fitting.

The row-level split is used only as a reference comparison. It is **not** presented as the original Week-5 validation design. The Week-5 model already used a client-level holdout.

The primary metric is **Precision@50**, because the practical use case is to prioritize a small review queue. ROC-AUC is reported as a secondary ranking metric.


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ============================================================
# SECTION 2 — HONEST VALIDATION AUDIT
#
# Before = row-level reference split
# After  = client-grouped split
#
# IMPORTANT:
# This uses the FINAL Week-5 target definition:
# went_dark = March gsc_clicks == 0
#
# It does NOT use the discarded click-quantile target.
# ============================================================

import numpy as np
import pandas as pd
import duckdb

from google.colab import userdata

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score


# ============================================================
# 1. Reconnect to the FlyRank warehouse
# ============================================================

RANDOM_STATE = 42
K = 50

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN was not found in Colab Secrets. "
        "Add your Hugging Face token as HF_TOKEN and run this cell again."
    )

con = duckdb.connect()

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf
    (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )
    """
)

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"

FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"
MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"


# ============================================================
# 2. Build the FINAL Week-5 modeling population
#
# February:
#   - eligibility universe
#   - February feature signals
#
# March:
#   - outcome window
#   - went_dark target
# ============================================================

feb_universe = con.execute(
    f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS imp_feb,
        SUM(gsc_clicks) AS clk_feb

    FROM {FEB}

    WHERE gsc_data_available

    GROUP BY
        client_hash_id,
        content_hash_id

    HAVING
        SUM(gsc_impressions) >= 100
        AND SUM(gsc_clicks) >= 3
    """
).df()


march_label = con.execute(
    f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS imp_mar,
        SUM(gsc_clicks) AS clk_mar

    FROM {MAR}

    WHERE gsc_data_available

    GROUP BY
        client_hash_id,
        content_hash_id
    """
).df()


# Merge February eligibility with March outcome
model_frame = feb_universe.merge(
    march_label,
    on=[
        "client_hash_id",
        "content_hash_id"
    ],
    how="left"
)


# Missing March observations = zero observed traffic
model_frame["imp_mar"] = model_frame["imp_mar"].fillna(0)
model_frame["clk_mar"] = model_frame["clk_mar"].fillna(0)


# FINAL Week-5 target definition
model_frame["went_dark"] = (
    model_frame["clk_mar"] == 0
).astype(int)


# ============================================================
# 3. Build February feature table
#
# These are the exact five Week-5 predictors.
# ============================================================

feb_features = con.execute(
    f"""
    SELECT
        client_hash_id,
        content_hash_id,

        AVG(gsc_impressions) AS gsc_impressions,
        AVG(gsc_avg_position) AS gsc_avg_position,
        AVG(ga4_sessions) AS ga4_sessions,
        AVG(ga4_users) AS ga4_users,
        AVG(ga4_engaged_sessions) AS ga4_engaged_sessions

    FROM {FEB}

    GROUP BY
        client_hash_id,
        content_hash_id
    """
).df()


FEATURES = [
    "gsc_impressions",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_users",
    "ga4_engaged_sessions"
]

TARGET = "went_dark"
GROUP = "client_hash_id"


# Final modeling dataset
modeling_data = feb_features.merge(
    model_frame[
        [
            "client_hash_id",
            "content_hash_id",
            TARGET
        ]
    ],
    on=[
        "client_hash_id",
        "content_hash_id"
    ],
    how="inner"
)


print("=== FINAL MODELING DATA ===")
print(f"Rows: {len(modeling_data):,}")
print(f"Clients: {modeling_data[GROUP].nunique():,}")
print(f"Target rate: {modeling_data[TARGET].mean():.3%}")

print("\nTarget distribution:")
print(
    modeling_data[TARGET]
    .value_counts()
    .sort_index()
)


# ============================================================
# 4. Define the Week-5 Logistic Regression pipeline
# ============================================================

def make_model():

    return Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(strategy="median")
            ),
            (
                "scaler",
                StandardScaler()
            ),
            (
                "model",
                LogisticRegression(
                    max_iter=1000,
                    class_weight="balanced",
                    random_state=RANDOM_STATE
                )
            )
        ]
    )


# ============================================================
# 5. Precision@K
# ============================================================

def precision_at_k(y_true, scores, k=50):

    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    k = min(k, len(y_true))

    top_k_idx = np.argsort(scores)[::-1][:k]

    return y_true[top_k_idx].mean()


# ============================================================
# 6. BEFORE — row-level reference split
#
# This is intentionally NOT described as Week-5's original split.
# It is only a reference design for the audit.
# ============================================================

before_train, before_test = train_test_split(
    modeling_data,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=modeling_data[TARGET]
)


X_before_train = before_train[FEATURES]
y_before_train = before_train[TARGET]

X_before_test = before_test[FEATURES]
y_before_test = before_test[TARGET]


before_model = make_model()

before_model.fit(
    X_before_train,
    y_before_train
)

before_scores = before_model.predict_proba(
    X_before_test
)[:, 1]


before_precision = precision_at_k(
    y_before_test,
    before_scores,
    K
)

before_auc = roc_auc_score(
    y_before_test,
    before_scores
)


before_client_overlap = len(
    set(before_train[GROUP])
    .intersection(
        set(before_test[GROUP])
    )
)


# ============================================================
# 7. AFTER — client-grouped split
# ============================================================

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE
)


after_train_idx, after_test_idx = next(
    gss.split(
        modeling_data,
        y=modeling_data[TARGET],
        groups=modeling_data[GROUP]
    )
)


after_train = modeling_data.iloc[
    after_train_idx
].copy()

after_test = modeling_data.iloc[
    after_test_idx
].copy()


X_after_train = after_train[FEATURES]
y_after_train = after_train[TARGET]

X_after_test = after_test[FEATURES]
y_after_test = after_test[TARGET]


after_model = make_model()

after_model.fit(
    X_after_train,
    y_after_train
)

after_scores = after_model.predict_proba(
    X_after_test
)[:, 1]


after_precision = precision_at_k(
    y_after_test,
    after_scores,
    K
)

after_auc = roc_auc_score(
    y_after_test,
    after_scores
)


after_client_overlap = len(
    set(after_train[GROUP])
    .intersection(
        set(after_test[GROUP])
    )
)


# ============================================================
# 8. BEFORE / AFTER comparison
# ============================================================

comparison = pd.DataFrame(
    [
        {
            "Validation":
                "Before — row-level reference",

            "Train rows":
                len(before_train),

            "Test rows":
                len(before_test),

            "Train clients":
                before_train[GROUP].nunique(),

            "Test clients":
                before_test[GROUP].nunique(),

            "Client overlap":
                before_client_overlap,

            "Precision@50":
                before_precision,

            "ROC-AUC":
                before_auc
        },

        {
            "Validation":
                "After — client-grouped",

            "Train rows":
                len(after_train),

            "Test rows":
                len(after_test),

            "Train clients":
                after_train[GROUP].nunique(),

            "Test clients":
                after_test[GROUP].nunique(),

            "Client overlap":
                after_client_overlap,

            "Precision@50":
                after_precision,

            "ROC-AUC":
                after_auc
        }
    ]
)


print("\n=== BEFORE / AFTER VALIDATION COMPARISON ===")

display(
    comparison.style.format(
        {
            "Precision@50": "{:.3f}",
            "ROC-AUC": "{:.3f}"
        }
    )
)


# ============================================================
# 9. Validation checks
# ============================================================

print("\n=== VALIDATION CHECKS ===")

assert after_client_overlap == 0

assert set(FEATURES).issubset(
    set(modeling_data.columns)
)

assert TARGET in modeling_data.columns

assert "clk_mar" not in FEATURES
assert "imp_mar" not in FEATURES

print("✓ Client-grouped split has zero client overlap.")
print("✓ February signals are used as model features.")
print("✓ March clicks define the went_dark outcome.")
print("✓ March outcome columns are excluded from X.")
print("✓ The same Logistic Regression pipeline is used in both comparisons.")
print("✓ Precision@50 is the primary ranking metric.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== FINAL MODELING DATA ===
Rows: 29,729
Clients: 32
Target rate: 5.150%

Target distribution:
went_dark
0    28198
1     1531
Name: count, dtype: int64

=== BEFORE / AFTER VALIDATION COMPARISON ===


,Validation,Train rows,Test rows,Train clients,Test clients,Client overlap,Precision@50,ROC-AUC
0,Before — row-level reference,23783,5946,31,30,29,0.360,0.769
1,After — client-grouped,22080,7649,25,7,0,0.160,0.695



=== VALIDATION CHECKS ===
✓ Client-grouped split has zero client overlap.
✓ February signals are used as model features.
✓ March clicks define the went_dark outcome.
✓ March outcome columns are excluded from X.
✓ The same Logistic Regression pipeline is used in both comparisons.
✓ Precision@50 is the primary ranking metric.


### Observation

The row-level reference split produced a measured **Precision@50 of 0.360** and **ROC-AUC of 0.769**. The client-grouped split produced a measured **Precision@50 of 0.160** and **ROC-AUC of 0.695**.

The key validation difference is the treatment of clients. The row-level reference split allowed the same clients to appear in both training and test, with **29 clients overlapping** between the two partitions. In contrast, the grouped evaluation kept clients entirely within one partition and had **0 client overlap**.

Under the grouped design, measured Precision@50 decreased from **0.360 to 0.160**, while ROC-AUC decreased from **0.769 to 0.695**. This indicates that the measured performance is sensitive to the validation design and that the row-level estimate was more optimistic than the client-grouped estimate.

The grouped result is therefore the preferred estimate for this task because it evaluates the model on clients that were not used during model fitting. This comparison is treated as evidence about **validation sensitivity**, not as proof that the model will generalize to every future client.

The Week-5 model used a client-level holdout, so the row-level result is included here only as a reference point for the audit. The main conclusion is that **client-level grouping is the more appropriate validation design for evaluating performance on unseen clients**.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## 3. Leakage Audit

A feature is considered potentially leaky when it contains information that would not have been available at the time the prediction was supposed to be made, or when it directly or indirectly contains information derived from the outcome.

For this model, the intended prediction point is the end of the February 2026 feature window. The outcome is measured during March 2026.

I therefore audit each feature against three questions:

1. **Time availability:** Was the feature calculated only from information available by the prediction point?
2. **Outcome contamination:** Does the feature contain March outcome information, directly or through a derived field?
3. **Identifier or aggregation leakage:** Could the feature encode the target through an aggregation or construction that uses information from the future or from the held-out evaluation population?

The Week-5 feature set consists of five February signals: `gsc_impressions`, `gsc_avg_position`, `ga4_sessions`, `ga4_users`, and `ga4_engaged_sessions`. March clicks are used to construct the `went_dark` outcome and are not included as model features. ([raw.githubusercontent.com](https://raw.githubusercontent.com/Rowan-ali/flyrank-ml-internship/main/work/notebooks/w05_model.ipynb))

The audit below checks the actual feature columns and their source periods rather than relying only on feature names.


In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ============================================================
# SECTION 3 — LEAKAGE AUDIT
# ============================================================

# ------------------------------------------------------------
# 1. Explicit feature / target inventory
# ------------------------------------------------------------

FEATURES = [
    "gsc_impressions",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_users",
    "ga4_engaged_sessions"
]

TARGET = "went_dark"

MARCH_OUTCOME_COLUMNS = [
    "clk_mar",
    "imp_mar",
    "gsc_clicks"
]

print("=== FEATURE INVENTORY ===")

feature_audit = pd.DataFrame(
    {
        "Feature": FEATURES,
        "Source period": ["February 2026"] * len(FEATURES),
        "Role": ["Model feature"] * len(FEATURES),
        "March outcome field?": ["No"] * len(FEATURES),
        "Target-derived?": ["No"] * len(FEATURES),
        "Leakage status": ["Pass — based on construction"] * len(FEATURES)
    }
)

display(feature_audit)


# ------------------------------------------------------------
# 2. Verify that March outcome columns are NOT model inputs
# ------------------------------------------------------------

feature_set = set(FEATURES)

march_overlap = feature_set.intersection(
    set(MARCH_OUTCOME_COLUMNS)
)

target_overlap = feature_set.intersection(
    {TARGET}
)

print("\n=== DIRECT LEAKAGE CHECK ===")

print(
    "March outcome columns included as features:",
    march_overlap
)

print(
    "Target included as a feature:",
    target_overlap
)

assert len(march_overlap) == 0
assert len(target_overlap) == 0

print("✓ No March outcome column is included in the feature set.")
print("✓ The target itself is not included in the feature set.")


# ------------------------------------------------------------
# 3. Verify feature availability in the February table
# ------------------------------------------------------------

feb_feature_columns = con.execute(
    f"""
    SELECT *
    FROM {FEB}
    LIMIT 1
    """
).df().columns.tolist()

missing_features = [
    f for f in FEATURES
    if f not in feb_feature_columns
]

print("\n=== SOURCE COLUMN CHECK ===")
print("Missing February feature columns:", missing_features)

assert len(missing_features) == 0

print("✓ All five model features exist in the February source data.")



# ------------------------------------------------------------
# 4. Verify feature source period
#
# The warehouse partition itself identifies the source period:
# month=2026-02.
# We do not assume a column named "date" exists.
# ------------------------------------------------------------

print("\n=== FEBRUARY FEATURE WINDOW CHECK ===")

print("Feature source:")
print("  February partition: month=2026-02")

feb_schema = con.execute(
    f"""
    DESCRIBE SELECT *
    FROM {FEB}
    """
).df()

print("\nAvailable source columns:")
display(
    feb_schema[["column_name", "column_type"]]
)

print("✓ Feature source is the February 2026 partition.")
print("✓ No date column was assumed for this check.")


# ------------------------------------------------------------
# 5. Verify March outcome source period
# ------------------------------------------------------------

print("\n=== MARCH OUTCOME WINDOW CHECK ===")

print("Outcome source:")
print("  March partition: month=2026-03")

mar_schema = con.execute(
    f"""
    DESCRIBE SELECT *
    FROM {MAR}
    """
).df()

print("\nAvailable March source columns:")
display(
    mar_schema[["column_name", "column_type"]]
)

print("✓ March data is used as the outcome window.")
print("✓ March outcome data is kept separate from the February features.")


# ------------------------------------------------------------
# 6. Check for suspicious feature names
# ------------------------------------------------------------

suspicious_terms = [
    "target",
    "label",
    "outcome",
    "march",
    "future",
    "next",
    "went_dark"
]

suspicious_features = [
    f
    for f in FEATURES
    if any(
        term in f.lower()
        for term in suspicious_terms
    )
]

print("\n=== FEATURE-NAME SANITY CHECK ===")

print(
    "Suspicious feature names:",
    suspicious_features
)

assert len(suspicious_features) == 0

print(
    "✓ No feature name indicates a direct "
    "target/future-period field."
)


# ------------------------------------------------------------
# 7. Final audit table
# ------------------------------------------------------------

final_leakage_audit = pd.DataFrame(
    [
        {
            "Audit":
                "Temporal availability",

            "Evidence":
                "All five predictors are sourced from "
                "the February 2026 partition.",

            "Status":
                "PASS"
        },

        {
            "Audit":
                "Direct target leakage",

            "Evidence":
                "went_dark is not included in FEATURES.",

            "Status":
                "PASS"
        },

        {
            "Audit":
                "March outcome leakage",

            "Evidence":
                "March outcome fields are excluded "
                "from FEATURES.",

            "Status":
                "PASS"
        },

        {
            "Audit":
                "Future-period feature leakage",

            "Evidence":
                "Feature construction reads the "
                "February partition only.",

            "Status":
                "PASS"
        }
    ]
)

print("\n=== FINAL LEAKAGE AUDIT ===")

display(final_leakage_audit)

=== FEATURE INVENTORY ===


,Feature,Source period,Role,March outcome field?,Target-derived?,Leakage status
0,gsc_impressions,February 2026,Model feature,No,No,Pass — based on construction
1,gsc_avg_position,February 2026,Model feature,No,No,Pass — based on construction
2,ga4_sessions,February 2026,Model feature,No,No,Pass — based on construction
3,ga4_users,February 2026,Model feature,No,No,Pass — based on construction
4,ga4_engaged_sessions,February 2026,Model feature,No,No,Pass — based on construction



=== DIRECT LEAKAGE CHECK ===
March outcome columns included as features: set()
Target included as a feature: set()
✓ No March outcome column is included in the feature set.
✓ The target itself is not included in the feature set.

=== SOURCE COLUMN CHECK ===
Missing February feature columns: []
✓ All five model features exist in the February source data.

=== FEBRUARY FEATURE WINDOW CHECK ===
Feature source:
  February partition: month=2026-02

Available source columns:


,column_name,column_type
0,report_date,DATE
1,client_hash_id,VARCHAR
2,content_hash_id,VARCHAR
3,client_has_gsc,BOOLEAN
4,client_has_ga4,BOOLEAN
5,gsc_data_available,BOOLEAN
6,ga4_data_available,BOOLEAN
7,gsc_impressions,BIGINT
8,gsc_clicks,BIGINT
9,gsc_sum_position,BIGINT


✓ Feature source is the February 2026 partition.
✓ No date column was assumed for this check.

=== MARCH OUTCOME WINDOW CHECK ===
Outcome source:
  March partition: month=2026-03

Available March source columns:


,column_name,column_type
0,report_date,DATE
1,client_hash_id,VARCHAR
2,content_hash_id,VARCHAR
3,client_has_gsc,BOOLEAN
4,client_has_ga4,BOOLEAN
5,gsc_data_available,BOOLEAN
6,ga4_data_available,BOOLEAN
7,gsc_impressions,BIGINT
8,gsc_clicks,BIGINT
9,gsc_sum_position,BIGINT


✓ March data is used as the outcome window.
✓ March outcome data is kept separate from the February features.

=== FEATURE-NAME SANITY CHECK ===
Suspicious feature names: []
✓ No feature name indicates a direct target/future-period field.

=== FINAL LEAKAGE AUDIT ===


,Audit,Evidence,Status
0,Temporal availability,All five predictors are sourced from the Febru...,PASS
1,Direct target leakage,went_dark is not included in FEATURES.,PASS
2,March outcome leakage,March outcome fields are excluded from FEATURES.,PASS
3,Future-period feature leakage,Feature construction reads the February partit...,PASS


### Observation

The leakage audit found **no identified direct temporal or target leakage** in the five audited model features.

All five predictors — `gsc_impressions`, `gsc_avg_position`, `ga4_sessions`, `ga4_users`, and `ga4_engaged_sessions` — are sourced from the **February 2026 partition**, while the `went_dark` outcome is constructed from the **March 2026 outcome window**. The audit confirmed that no March outcome field and no target field is included in the model feature list.

The feature-name sanity check also found no feature whose name directly indicates a target, label, future period, or `went_dark` field.

These checks support the measured conclusion that the audited feature construction separates the February prediction inputs from the March outcome. However, this is a **construction-level leakage audit**: it does not prove that every possible form of indirect or upstream data contamination is impossible.

Based on the checks performed, **no direct leakage was identified in the audited feature set**.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## 4. Claim Rewrite

The validation audit changed how I would communicate the Week-5 model results.

The original modeling work was designed as a decision-support ranking task, with Precision@50 used to measure how many relevant records appeared in the top 50 ranked observations. The model also used a client-level holdout in the original Week-5 evaluation. ([raw.githubusercontent.com](https://raw.githubusercontent.com/Rowan-ali/flyrank-ml-internship/main/work/notebooks/w05_model.ipynb))

The audit shows why the validation design needs to be stated alongside the metric. In the row-level reference split, the measured Precision@50 was 0.360 and ROC-AUC was 0.769. Under the client-grouped split, Precision@50 was 0.160 and ROC-AUC was 0.695, with zero client overlap.

I therefore avoid presenting the higher row-level result as evidence of general model performance. The client-grouped result is the more appropriate estimate for the intended question of performance on unseen clients.

### Claim rewrites

| Earlier / stronger interpretation                               | Safer claim                                                                                                                                                                              |
| --------------------------------------------------------------- | ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| “The model performs well at identifying high-priority records.” | “The model produced a measured Precision@50 of 0.160 and ROC-AUC of 0.695 under the client-grouped evaluation.”                                                                          |
| “The model is better than the baseline.”                        | “Under the measured evaluation setup, the model can be used as a decision-support ranking signal; any comparison with the baseline should use the same client-grouped split and metric.” |
| “The model can predict which content will go dark.”             | “The model provides a directional ranking of content associated with the `went_dark` outcome in the evaluated client holdout.”                                                           |
| “The features predict future performance.”                      | “The February feature signals were associated with the March outcome in this evaluated sample.”                                                                                          |
| “The model generalizes to clients.”                             | “The grouped holdout provides a measured estimate on clients not used during model fitting.”                                                                                             |
| “Higher feature values cause the outcome to change.”            | “The fitted coefficients describe directional associations within the model; they do not establish causation.”                                                                           |

### Public-safe conclusion

The model is best described as a **decision-support ranking tool** rather than a guaranteed predictor. In the client-grouped audit, it achieved a measured Precision@50 of 0.160 and ROC-AUC of 0.695 on held-out clients. The lower grouped performance relative to the row-level reference indicates that the measured result is sensitive to validation design.

The evidence supports using the model to prioritize records for review and further investigation. It does not support a claim that the model will reliably identify every future `went_dark` case or generalize universally to new clients.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.